In [ ]:

import numpy as np
from typing import Optional

def add_poisson_noise(
    counts: np.ndarray,
    lam_scale: float = 1.0,
    random_state: Optional[int] = None
) -> np.ndarray:

    if random_state is not None:
        np.random.seed(random_state)
    
    noisy_counts = np.random.poisson(lam_scale * counts)
    return noisy_counts

def add_nb_noise(
    counts: np.ndarray,
    dispersion: float = 10.0,
    random_state: Optional[int] = None
) -> np.ndarray:
   
    if random_state is not None:
        np.random.seed(random_state)
    mu = counts.astype(float)
    n = dispersion

    p = n / (n + mu)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    noisy_counts = np.random.negative_binomial(n, p)
    return noisy_counts


if __name__ == "__main__":
    
    #     params = [
    #         {"lam_scale": 0.1, "dispersion": 10},
    #         {"lam_scale": 0.3, "dispersion": 1},
    #         {"lam_scale": 0.5, "dispersion": 0.1},
    #     ]
  
    # adata_omics1 = sc.read_h5ad("/00_data/02_Human_tonsil_slice1/adata_RNA.h5ad")
    # adata_omics1.X = add_poisson_noise(adata_omics1.X.toarray(), lam_scale=1.0, random_state=42)

    adata_omics2 = sc.read_h5ad("00_data/12_Slide_tags/adata_ATAC.h5ad")
    adata_omics2.X = add_nb_noise(adata_omics2.X.toarray(), dispersion=0.01, random_state=42)

    # adata_omics1.write("/00_data/14_add_noise_and_mask_dataset/02_Human_tonsil_slice1_noise_04/adata_RNA.h5ad")

    adata_omics2.write("/00_data/14_add_noise_and_mask_dataset/12_Slide_tags_noise_04/adata_ATAC.h5ad")


In [ ]:
# Rescue all-zero cells (introduced by noise/masking) by randomly setting one feature to 1, then filter features with min_cells=3.
# This prevents downstream analysis (e.g. LSI/PCA) from failing due to empty cells.

import scanpy as sc
import numpy as np
import scipy.sparse as sp

adata = sc.read_h5ad("/00_data/14_add_noise_and_mask_dataset/12_Slide_tags_mask_09/adata_ATAC.h5ad")

if sp.issparse(adata.X):
    cell_sums = np.array(adata.X.sum(axis=1)).ravel()
else:
    cell_sums = np.array(adata.X.sum(axis=1))
zero_cells = np.where(cell_sums == 0)[0]
print(f"There are {len(zero_cells)} all-zero cells that need to be rescued.")

n_cells, n_feats = adata.X.shape
rng = np.random.default_rng(seed=2025)  

if len(zero_cells) > 0:
    if sp.issparse(adata.X):
        Xlil = adata.X.tolil()
        for i in zero_cells:
            j = rng.integers(0, n_feats)
            Xlil[i, j] = 1
        adata.X = Xlil.tocsr()
    else:
        for i in zero_cells:
            j = rng.integers(0, n_feats)
            adata.X[i, j] = 1

if sp.issparse(adata.X):
    new_sums = np.array(adata.X.sum(axis=1)).ravel()
else:
    new_sums = np.array(adata.X.sum(axis=1))
print("Number of all-zero rows after rescue:", np.sum(new_sums == 0))

adata_omics2 = adata
sc.pp.filter_genes(adata_omics2, min_cells=3)   
if 'X_lsi' not in adata_omics2.obsm.keys():
    sc.pp.highly_variable_genes(adata_omics2, flavor="seurat_v3", n_top_genes=3000)

adata.write("/00_data/14_add_noise_and_mask_dataset/12_Slide_tags_mask_09/adata_ATAC.h5ad")

